# Первая лабораторная работа: Fusion на датасете PMEmo

Будем решать проблему характеристики песен по тексту и аудио

## Импорты и константы

In [3]:
import re
import warnings
from datetime import datetime
from pathlib import Path
from pprint import pp

import librosa
import pandas as pd
import torch
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from transformers import (
    BertModel,
    BertTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Model,
)

warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
SR = 16000
RANDOM_STATE = 42

In [5]:
AUDIO_CORPUS_PATH = Path("data/PMEmo2019/chorus")
LYRICS_CORPUS_PATH = Path("data/PMEmo2019/lyrics")
ANNOTATIONS = Path("data/PMEmo2019/annotations/static_annotations.csv")

In [6]:
annotations_df = pd.read_csv(ANNOTATIONS, sep=",", index_col="musicId")
annotations_df.head()

,Arousal(mean),Valence(mean)
musicId,,
1,0.4000,0.5750
4,0.2625,0.2875
5,0.1500,0.2000
6,0.5125,0.3500
7,0.7000,0.7250


In [7]:
audio_paths = []
text_paths = []
annotations = []

for root, dirs, files in AUDIO_CORPUS_PATH.walk():
    for file in files:
        audio_path = root / file
        try:
            music_id = int(str(Path(file).with_suffix("")))
            annotation = annotations_df.loc[music_id]
            if isinstance(annotation, pd.DataFrame):
                annotation = annotation.mean()

            text_path = LYRICS_CORPUS_PATH / Path(file).with_suffix(".lrc")
            if not text_path.exists():
                continue

            audio_paths.append(audio_path)
            text_paths.append(text_path)
            annotations.append(annotation.tolist())
        except KeyError:
            pass

In [8]:
print(f"Кол-во аудиозаписей: {len(audio_paths)}\nКол-во лейблов: {len(annotations)}")

Кол-во аудиозаписей: 606
Кол-во лейблов: 606


In [9]:
pp(list(zip(audio_paths[:5], annotations[:5])))

[(PosixPath('data/PMEmo2019/chorus/851.mp3'), [0.4625, 0.4125]),
 (PosixPath('data/PMEmo2019/chorus/888.mp3'), [0.675, 0.675]),
 (PosixPath('data/PMEmo2019/chorus/28.mp3'), [0.4875, 0.575]),
 (PosixPath('data/PMEmo2019/chorus/765.mp3'), [0.8, 0.7625]),
 (PosixPath('data/PMEmo2019/chorus/964.mp3'), [0.5375, 0.6])]


In [ ]:
class EntryInfo:
    def __init__(self, audio_path, lyrics_path, annotations_path):
        self.audio_path = audio_path
        self.lyrics_path = lyrics_path
        self.annotations_path = annotations_path

        self.timestamps, self.lyrics = self.get_lyrics()
        self.audio = self.get_audio()
        self.annotations = self.get_annotations()

    def get_lyrics(self):
        def read_file(file_path):
            with open(file_path) as f:
                file_list = f.readlines()
            return file_list

        def clean_lyrics(line):
            line = line.strip()
            return line

        def convert_to_secs(timestamp):
            timestamp = timestamp.split(".")[0].split(":")
            seconds = int(timestamp[0]) * 60 + int(timestamp[1])
            return seconds

        lyrics = read_file(self.lyrics_path)

        timestamp_regex = r"\d{2}:\d{2}\.\d{2}|$"
        lyrics_regex = r"\[\d{2}:\d{2}\.\d{2}\]|$"

        timestamps = []
        lyrics_lines = []
        for line in lyrics:
            timestamp = re.findall(timestamp_regex, line)[0]
            text = re.sub(lyrics_regex, "", line)

            if not timestamp:
                continue

            timestamps.append(convert_to_secs(timestamp))
            lyrics_lines.append(clean_lyrics(text))

        return timestamps, lyrics_lines

    def get_audio(self):
        audio, _ = librosa.load(self.audio_path, sr=SR)
        
        audio_timestamps = []
        for timestamp_start, timestamp_end in zip(self.timestamps[:-1], self.timestamps[1:]):
            audio_frame = audio[int(timestamp_start * SR):int(timestamp_end * SR)]
            if len(audio_frame) > 0:
                audio_timestamps.append(audio_frame)

        return audio_timestamps

    def get_annotations(self):
        pass
        # def read_csv(self, file_path):
        #     df = pd.read_csv(ANNOTATIONS, sep=",", index_col="musicId")
        #     return df

In [20]:
entry = EntryInfo(
    "data/PMEmo2019/chorus/1.mp3",
    "data/PMEmo2019/lyrics/1.lrc", 
    "data/PMEmo2019/annotations/dynamic_annotations.csv"
)

audio_frames = entry.get_audio()
print(len(audio_frames))
print(audio_frames[0])

# for frame in audio_frames:
#     if len(frame) == 0:
#         print(frame)

7
[0.10653649 0.09110215 0.0843449  ... 0.04378288 0.03659716 0.03537604]


In [12]:
def read_lrc(text_path):
    def lrc_preprocess(line):
        line = line.strip()
        line = re.sub(r"\[.+\]", "", line)
        return line

    with open(text_path) as f:
        lrc_list = f.readlines()
    lrc_list = [lrc_preprocess(lrc_string) for lrc_string in lrc_list]
    lrc_list = list(filter(len, lrc_list))

    return lrc_list

[ ] Переделать сюда, чтобы окна брались по строкам из песен, а не по фиксированному значению
[ ] Переделать сюда, чтобы лейблы внутри окон бли динамическими

In [13]:
def make_audio_windows(audio, len_window_ms=30):
    len_window_samples = int(len_window_ms / 1000 * SR)
    windows = []
    for window in range(0, len(audio), len_window_samples):
        if (window + len_window_samples) <= len(audio):
            windows.append(audio[window : window + len_window_samples])
        else:
            windows.append(audio[window:])
    return windows

## Извлечение признаков

### wav2vec2

In [43]:
wav2vec2_model = Wav2Vec2Model.from_pretrained(
    "facebook/wav2vec2-base-960h"
)
wav2vec2_model = wav2vec2_model.to(device)
wav2vec2_feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(
    "facebook/wav2vec2-base-960h"
)

Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [44]:
wav2vec2_embeddings = []
for audio_path in tqdm(audio_paths):
    audio, _ = librosa.load(audio_path, sr=SR)

    # -> [[window], [window]]
    audio = make_audio_windows(audio)

    inputs = wav2vec2_feature_extractor(
        audio,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )
    inputs = inputs.to(device)

    with torch.inference_mode():
        outputs = wav2vec2_model(**inputs)

    features = outputs.last_hidden_state
    embedding = features.mean(dim=1).to("cpu")

    wav2vec2_embeddings.append(embedding)

100%|██████████| 606/606 [05:50<00:00,  1.73it/s]


In [45]:
torch.save(wav2vec2_embeddings, "wav2vec2_embeddings.pt")

### BERT

In [46]:
# Load pre-trained BERT
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertModel.from_pretrained("bert-base-uncased")

bert_model = bert_model.to(device)
# bert_model.eval()

In [47]:
bert_embeddings = []
for text_path in tqdm(text_paths):
    # -> [[lyrics line], [lyrics line]]
    text = read_lrc(text_path)

    inputs = bert_tokenizer(
        text,
        return_tensors="pt",
        padding=True
    )
    inputs = inputs.to(device)

    with torch.inference_mode():
        outputs = bert_model(
            **inputs, 
            output_hidden_states=True
        )

    # The final layer's [CLS] representation
    features = outputs.last_hidden_state
    embedding = features.mean(dim=1).to("cpu")

    bert_embeddings.append(embedding)

100%|██████████| 606/606 [01:52<00:00,  5.38it/s]


In [48]:
torch.save(bert_embeddings, "bert_embeddings.pt")

## Регрессор по отдельности

### Аудио

In [49]:
wav2vec2_embeddings = torch.load("wav2vec2_embeddings.pt")

In [50]:
# Усреднение по окнам для каждого аудио
wav2vec2_emb_audio_mean = []
for audio_emb in wav2vec2_embeddings:
    wav2vec2_emb_audio_mean.append(audio_emb.mean(dim=0))

In [51]:
wav2vec2_embeddings_train, wav2vec2_embeddings_test, \
    wav2vec2_annotations_train, wav2vec2_annotations_test = train_test_split(
        wav2vec2_emb_audio_mean,
        annotations, 
        test_size=0.2, 
        random_state=RANDOM_STATE
)

In [52]:
wav2vec2_embeddings_train[0].shape, wav2vec2_annotations_train[0]

(torch.Size([768]), [0.4125, 0.35])

In [53]:
linreg = LinearRegression()
linreg.fit(wav2vec2_embeddings_train, wav2vec2_annotations_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float32](2, 768)","[[-25.34, 2.62,-10.87,..., 48.56,-81.28, 36.6 ], [ -1.34,-33. ,-27.7 ,...,-14.56,-54.43, -0.63]]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.","ndarray[float32](2,)","[-18.32,-78.29]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,768
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,480
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float32](484,)","[2.7 ,2.21,1.34,...,0. ,0. ,0. ]"


In [54]:
linreg_preds = linreg.predict(wav2vec2_embeddings_test)
print(r2_score(wav2vec2_annotations_test, linreg_preds))

-2.8857905864715576


### Текст

In [55]:
bert_embeddings = torch.load("bert_embeddings.pt")

In [56]:
# Усреднение по окнам для каждого текста
bert_emb_audio_mean = []
for text_emb in bert_embeddings:
    bert_emb_audio_mean.append(text_emb.mean(dim=0))

In [57]:
bert_embeddings_train, bert_embeddings_test, \
    bert_annotations_train, bert_annotations_test = train_test_split(
        bert_emb_audio_mean,
        annotations, 
        test_size=0.2, 
        random_state=RANDOM_STATE
)

In [58]:
bert_embeddings_train[0].shape, bert_annotations_train[0]

(torch.Size([768]), [0.4125, 0.35])

In [59]:
linreg = LinearRegression()
linreg.fit(bert_embeddings_train, bert_annotations_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float32](2, 768)","[[-0.21, 0.12, 0.27,..., 0.4 , 0.42,-0.65], [-0.6 , 0.34, 0.17,...,-0.06, 0.24,-0.28]]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.","ndarray[float32](2,)","[-7.16,-2.62]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,768
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,479
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float32](484,)","[20.33,17.51,14.67,..., 0. , 0. , 0. ]"


In [60]:
linreg_preds = linreg.predict(bert_embeddings_test)
print(r2_score(bert_annotations_test, linreg_preds))

-7.20620059967041


## Early Fusion

## Intermideate Fusion

## Late Fusion